# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
md = dataset.metadata  # md is a DatasetMetadata object

print(f"Dataset Title: {md.name}")
print(f"Description: {md.description}\n")
print(f"Published: {md.date_published}")
print(f"Version: {md.version}")
print("License:", md.license)


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's examine which top-level RecordSets are defined in the Croissant schema, along with their fields and columns. All references use the canonical `@id`.

In [ ]:
# Get a list of record sets and show their @id and labels
print("Record sets in this dataset:")
record_sets = md.record_sets
for i, rec in enumerate(record_sets):
    print(f"[{i}] @id: {rec.id}")
    print(f"    name: {rec.name}")
    print(f"    description: {rec.description}")
    print(f"    Number of fields: {len(rec.fields)}\n")

# Display fields within each record set
for i, rec in enumerate(record_sets):
    print(f"Fields in RecordSet '{rec.name}' (id: {rec.id}):")
    for f in rec.fields:
        col_spec = f.column.id if f.column is not None else None
        print(f"  • Field @id: {f.id}\n      name: {f.name}\n      type: {f.data_type}\n      column @id: {col_spec}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We will extract all data tables and demonstrate with the main protein annotation record set.

In [ ]:
# List the record set ids for convenience
record_set_ids = [rec.id for rec in record_sets]
# Map: name to id, too
record_set_names = {rec.name: rec.id for rec in record_sets}

dataframes = {}
for rec_id in record_set_ids:
    # Load all rows from this record set
    print(f"Loading records for record set: {rec_id}")
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df
    print(f"  Shape: {df.shape}\n")

# Let's pick the first record set as the primary one for EDA
main_rec_id = record_set_ids[0]
main_df = dataframes[main_rec_id]
print("Available columns:")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's select a numeric field (e.g. `cr:field_coverage_percent` if present) for filtering and normalization. We'll group by another relevant attribute, such as the protein accession (if present). All field and column names are referenced by their `@id`.

*(Note: Please adjust `numeric_field_id` and `group_field_id` to your dataset fields as shown above.)*

In [ ]:
# Example: Choose a numeric field from fields in the record set
# Replace these placeholder IDs with the actual `@id` of the numeric field and group field you want to analyze
fields = [f for f in record_sets[0].fields]
numeric_fields = [(f.name, f.id) for f in fields if f.data_type in ["schema:Float","schema:Integer","schema:Number"]]
print("Numeric field candidates:", numeric_fields)

# For demonstration, pick the first numeric one 
if numeric_fields:
    numeric_field_id = numeric_fields[0][1]  # e.g. 'cr:field_coverage_percent'
    print(f"Using numeric field: {numeric_field_id}")
else:
    raise ValueError("No numeric field detected in first record set!")

group_field_candidates = [(f.name, f.id) for f in fields if f.data_type in ["schema:Text"]]
print("Groupable field candidates:", group_field_candidates)
group_field_id = group_field_candidates[0][1] if group_field_candidates else None
print(f"Grouping by: {group_field_id}")

# Remove NaN for analysis
df = main_df.dropna(subset=[numeric_field_id])

# Filtering
threshold = df[numeric_field_id].mean() if not df[numeric_field_id].empty else 0
print(f"Applying threshold {threshold:.2f} to field {numeric_field_id}")
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} total rows")
print(filtered_df[[numeric_field_id]].head())

# Normalize
filtered_df[numeric_field_id + "_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"\nGrouped data by {group_field_id} (showing means):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We will plot the distribution of the selected numeric field and, if available, a bar plot grouped by a key attribute.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

if group_field_id and group_field_id in filtered_df.columns:
    # Top 10 groups by mean value
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(10)
    plt.figure(figsize=(10,4))
    sns.barplot(x=grouped.values, y=grouped.index)
    plt.title(f"Top 10 {group_field_id} by mean {numeric_field_id}")
    plt.xlabel(f"Mean {numeric_field_id}")
    plt.ylabel(group_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook has walked through loading the FAIR² mass spectrometry dataset via its Croissant schema using `mlcroissant`, inspected its available record sets and fields, and performed sample exploratory data analysis and visualization. These steps can be adapted for further analysis or integration into machine learning pipelines as needed.